In [87]:
from datetime import datetime, timedelta
import re

In [88]:
# Tools

def parse_date(date: str):
    try:
        dt = datetime.strptime(date, "%Y-%m-%d")
        return dt.year, dt.month, dt.day
    except:
        raise ValueError(f"Invalid date format: {date}")



In [ ]:
class Library:
    borrowed_books = set()

    def __init__(self,name):
        self.name = name

    
    def book_available(self,Book):
        return Book.id not in Library.borrowed_books


    @classmethod
    def update_borrowed_books_on_return(cls,Book,Customer):

        if Book.id in cls.borrowed_books:
            cls.borrowed_books.remove(Book.id)
            Customer.cust_books_borrowed.pop(Book.id, None)
            return True
        else:
            raise KeyError(f'Book {Book.id} is not in list of borrowed books. Book must be borrowed before being returned')
        
    
    @classmethod
    def update_borrowed_books_on_loan(cls,Book):

        if Book.id not in cls.borrowed_books:
            cls.borrowed_books.add(Book.id)
            return True
        else:
            raise KeyError(f'Book {Book.id} is in list of borrowed books. Book must be return before being borrowed')


In [90]:
class Book:
    books_id = set()
    
    def __init__(self, id, title, author, resume, book_kind, restriction_age=None):
        if id in Book.books_id:
            raise KeyError(f"ID {id} déjà utilisé.")
        self.id = id 
        self.title = title
        self.author = author
        self.borrow_date = None # On stocke la date sur le livre pour simplifier
        Book.books_id.add(id)
    

In [91]:
class Customer:

    def __init__(self,name : str , first_name: str, birth_day : str, phone_number: str, address: str, email : str):
        self.name = name
        self.first_name = first_name
        self.birth_day = birth_day
        self.phone_number = phone_number
        self.address = address
        self.email = email
        self.cust_books_borrowed = {}
        self.penalty = 0



    def loan(self,book,date):

        if book.id in Library.borrowed_books:
            print(f"Refusé : {book.title} est déjà pris.")
            return

        if self.penalty == 0:
            year, month, day = parse_date(date)

            book.borrow_date = datetime(year,month,day)

            Library.update_borrowed_books_on_loan(book)

            self.cust_books_borrowed[book.id] = book.borrow_date
            print(f"{book.title} was borrowed on {book.borrow_date}")
        else:
            print(f"Loan denied : {self.first_name} has a penalty of {self.penalty} days.")

    def return_book(self,book,date,loan):

        if book.borrow_date is None:
            raise ValueError(f"Book '{book.title}' is not borrowed.")
        
        year, month, day = parse_date(date)
        return_date = datetime(year,month,day)
        
        timelapse = return_date - book.borrow_date
        if timelapse.days < 0:
            raise ValueError(f" {book.return_date} is before {book.borrow_date}")
        
        if timelapse.days > 15:
            self.penalty += (timelapse.days - 15)
            print(f"Late {timelapse.days - 15} days. Penalty is now {self.penalty} days.")

        


        self.cust_books_borrowed.pop(book.id, None)
        Library.update_borrowed_books_on_return(book, self)
        book.borrow_date = None
        print(f"{book.title} was return on {return_date.date()}")





In [92]:
    
class Author:
    '''Author'''
    authors_id = set()

    def __init__(self, id, name : str , first_name: str, biography :str):
        self.id = id
        self.update_authors(self.id)
        self.name = name
        self.first_name = first_name
        self.biography = biography
        self.books = set()
        

    @classmethod
    def update_authors(cls,id):
        if type(id) is not int:
            raise ValueError(f'ID {id} must be an int.')
        if id not in cls.authors_id:
            cls.authors_id.add(id)
        else:
            raise KeyError(f'Author {id} is already in list of authors. Id must be unique')
        return True

    
    

In [93]:

book_lotr = Book(101, "Le Seigneur des Anneaux", "Tolkien", "...", "Fantasy")
client1 = Customer("Dupont", "Jean", "1995-05-15", "06", "Paris", "jean@email.com")


client1.loan(book_lotr, "2026-04-15")
client1.return_book(book_lotr, "2026-05-15", loan1)


author1 = Author(8,"Tolkien", "J.R.R.", "Écrivain et philologue britannique, célèbre pour Le Seigneur des Anneaux.")
author2 = Author(9,"Orwell", "George", "Écrivain anglais dont l'œuvre porte la marque de ses engagements.")
author3 = Author(11,"Hugo", "Victor", "Poète, dramaturge, écrivain et romancier français.")


# ❌ ATTENTION: doublon ID supprimé → changer ID
book_1984 = Book(102, "1984", "George Orwell", "Un monde dystopique sous la surveillance de Big Brother.", "Dystopie", 15)
book_miserables = Book(103, "Les Misérables", "Victor Hugo", "Le destin de Jean Valjean dans la France du XIXe siècle.", "Classique", 10)
book_animal_farm = Book(104, "La Ferme des Animaux", "George Orwell", "Une satire de la révolution russe à travers une ferme.", "Fable", 12)


client1 = Customer("Dupont", "Jean", "1995-05-15", "0612345678", "12 rue de la Paix, Paris", "jean.dupont@email.com")
client2 = Customer("Lefebvre", "Marie", "2008-11-20", "0788990011", "5 avenue des Roses, Toulouse", "marie.l@email.com")


client1.loan(book_lotr, "2026-04-15")
client1.return_book(book_lotr, "2026-06-15", loan1) 

client2.loan(book_lotr, "2026-05-01")
client2.return_book(book_lotr, "2028-05-01",loan1)

print(f"Client 1 penalty: {client1.penalty} days")
print(f"Client 2 penalty: {client2.penalty} days")


Le Seigneur des Anneaux was borrowed on 2026-04-15 00:00:00
Late 15 days. Penalty is now 15 days.
Le Seigneur des Anneaux was return on 2026-05-15
Le Seigneur des Anneaux was borrowed on 2026-04-15 00:00:00
Late 46 days. Penalty is now 46 days.
Le Seigneur des Anneaux was return on 2026-06-15
Le Seigneur des Anneaux was borrowed on 2026-05-01 00:00:00
Late 716 days. Penalty is now 716 days.
Le Seigneur des Anneaux was return on 2028-05-01
Client 1 penalty: 46 days
Client 2 penalty: 716 days
